# Notebook de Modelagem e Teste dos Dados

Neste notebook será realizada a etapa de avaliação dos dados já preparados no notebook de preparação (`data_prep.ipynb`).

Aqui vamos testar o desempenho dos nossos dados com **Random Forest** como modelo inicial de referência, por ser uma abordagem robusta e eficiente para classificação. A ideia é verificar se, com as features já tratadas, normalizadas e separadas em treino, validação e teste, conseguimos obter uma boa acurácia e um comportamento consistente na predição das classes de Libras.

Caso a acurácia do Random Forest não seja satisfatória, ou se houver espaço para melhoria, partiremos para o **XGBoost**, que costuma oferecer maior poder de ajuste e melhor desempenho em problemas de classificação mais complexos.

O foco deste notebook é comparar o desempenho dos modelos, validar os resultados com métricas adequadas e escolher a estratégia mais eficiente para a etapa final da pipeline.

## Fluxo passo a passo

1. **Carregar os dados já preparados**
   - Utilizar os arquivos gerados no `data_prep.ipynb`, como `train.csv`, `validation.csv` e `test.csv`.
   - Garantir que as features estejam padronizadas e prontas para treinamento.

2. **Entender a base final de entrada**
   - Conferir o formato dos dados.
   - Separar corretamente variáveis de entrada (`X`) e rótulo alvo (`y`).

3. **Treinar um modelo base com Random Forest**
   - Usar o conjunto de treino para ajustar o modelo.
   - Avaliar o resultado inicial na validação.

4. **Medir a performance com métricas de classificação**
   - Verificar acurácia, matriz de confusão e outras métricas relevantes.
   - Analisar se o modelo está generalizando bem.

5. **Decidir o próximo modelo com base no resultado**
   - Se a acurácia do Random Forest for boa, seguir com ele como solução principal.
   - Se o resultado ficar abaixo do esperado, testar o **XGBoost** como alternativa mais forte.

6. **Comparar Random Forest e XGBoost**
   - Avaliar qual modelo apresenta melhor desempenho geral.
   - Considerar não apenas acurácia, mas também equilíbrio entre classes e robustez.

7. **Validar o modelo escolhido no conjunto de teste**
   - Fazer a avaliação final apenas após a escolha do melhor modelo.
   - Registrar os resultados para uso na pipeline.

8. **Salvar conclusões e próximos passos**
   - Documentar o modelo selecionado.
   - Indicar possíveis melhorias futuras na preparação dos dados ou no ajuste de hiperparâmetros.

### Imports

In [7]:
#Visualização 
import matplotlib.pyplot as plt
import seaborn as sns


#Organização dos dados
import pandas as pd 
from pathlib import Path

#Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

#Caminhos dos dados
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREPARED_DATA_DIR = PROJECT_ROOT / "data" / "data_output" / "prepared"

train_path = PREPARED_DATA_DIR / "train.csv"
test_path = PREPARED_DATA_DIR / "test.csv"
validate_path = PREPARED_DATA_DIR / "validation.csv"

### 1. Carregar Dados de Treino Já preparados

In [8]:
train_df = pd.read_csv(train_path)

train_df

,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z19,x20,y20,z20,source_file,dist_polegar_pulso,dist_indicador_pulso,dist_medio_pulso,dist_anelar_pulso,dist_minimo_pulso
0,G,0.0,0.0,0.0,0.885596,-0.379024,-0.200417,0.908502,-0.499526,0.025278,...,0.706030,-0.267319,0.411901,0.808504,datasetRGUC (1).csv,-0.043571,1.013427,-1.718647,-1.335493,-1.229031
1,V,0.0,0.0,0.0,-0.393007,-0.952717,0.391918,-0.300160,-0.490160,0.299965,...,0.201830,0.067611,0.268011,0.175307,datasetLOV.csv,-1.008475,0.584140,0.847467,-0.714722,-0.987711
2,R,0.0,0.0,0.0,-0.473423,0.208982,-0.496832,-0.391038,0.119701,-0.442205,...,-0.360021,-0.107682,0.620514,-0.342409,datasetRGUC (1).csv,-1.022960,0.682610,0.926278,-1.102736,-1.360196
3,B,0.0,0.0,0.0,0.805840,-0.164669,0.593794,0.750551,-0.275872,0.815293,...,0.361822,-0.064971,-0.734850,-0.066206,datasetNBQ (1).csv,-1.034299,0.282986,0.810639,1.162422,0.415489
4,F,0.0,0.0,0.0,0.982975,-0.408690,0.349499,0.902986,-0.437203,0.481304,...,-0.939653,-0.538136,-0.695819,-1.188053,datasetFSY (1).csv,-0.064272,-0.797495,1.021822,1.450210,0.482550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19206,L,0.0,0.0,0.0,-0.532397,0.285683,-0.615860,-0.694583,0.331221,-0.798556,...,0.008494,0.136129,0.264246,0.262083,datasetLOV.csv,-0.361606,0.970110,-1.406881,-0.925718,-0.978689
19207,V,0.0,0.0,0.0,-0.591951,0.191130,-0.024392,-0.542013,0.078077,-0.009446,...,-0.627424,-0.155977,0.333093,-0.707070,datasetLOV.csv,-1.100008,0.686085,0.861284,-0.623000,-0.968830
19208,S,0.0,0.0,0.0,-0.836652,-0.077649,-0.473066,-0.850657,-0.269808,-0.601322,...,-0.438485,0.299464,-0.336859,-0.071865,datasetFSY (1).csv,-0.475210,-0.395280,-0.274583,0.104441,-0.103833
19209,E,0.0,0.0,0.0,-0.761334,0.057181,-0.539851,-0.874414,-0.226355,-0.498550,...,1.153556,0.113456,-0.622177,1.210633,datasetAMEP (1).csv,-0.578976,-0.914588,-0.369708,0.250915,0.220234


In [9]:
# Seleciona os pontos 0, 4, 8, 12, 16 e 20
# e mantém as três dimensões de cada um: x, y e z.
indices_selecionados = [0, 4, 8, 12, 16, 20]

colunas_landmarks = [
    f"{dim}{idx}"
    for idx in indices_selecionados
    for dim in ("x", "y", "z")
]

# Colunas de distância que já existem no dataset.
colunas_dist = [col for col in train_df.columns if "dist" in col.lower()]

# Junta tudo sem duplicar e mantendo a ordem.
colunas_selecionadas = []
for col in colunas_landmarks + colunas_dist:
    if col in train_df.columns and col not in colunas_selecionadas:
        colunas_selecionadas.append(col)

x_train = train_df[colunas_selecionadas].copy()
y_train = train_df["label"].copy()

print("Colunas selecionadas:", colunas_selecionadas)
print("Classe alvo (y):", y_train.name)
x_train.head()

Colunas selecionadas: ['x0', 'y0', 'z0', 'x4', 'y4', 'z4', 'x8', 'y8', 'z8', 'x12', 'y12', 'z12', 'x16', 'y16', 'z16', 'x20', 'y20', 'z20', 'dist_polegar_pulso', 'dist_indicador_pulso', 'dist_medio_pulso', 'dist_anelar_pulso', 'dist_minimo_pulso']
Classe alvo (y): label


,x0,y0,z0,x4,y4,z4,x8,y8,z8,x12,...,y16,z16,x20,y20,z20,dist_polegar_pulso,dist_indicador_pulso,dist_medio_pulso,dist_anelar_pulso,dist_minimo_pulso
0,0.0,0.0,0.0,1.150640,-0.300642,0.259327,0.942919,-0.895598,0.462291,0.196231,...,0.361833,0.705068,-0.267319,0.411901,0.808504,-0.043571,1.013427,-1.718647,-1.335493,-1.229031
1,0.0,0.0,0.0,0.123700,-0.146464,0.057263,-0.145548,-0.836115,0.865206,0.499102,...,-0.006437,0.160247,0.067611,0.268011,0.175307,-1.008475,0.584140,0.847467,-0.714722,-0.987711
2,0.0,0.0,0.0,0.074920,-0.094994,-0.335548,-0.443577,-0.811484,-0.076570,-0.631682,...,0.296607,-0.276558,-0.107682,0.620514,-0.342409,-1.022960,0.682610,0.926278,-1.102736,-1.360196
3,0.0,0.0,0.0,0.539247,-0.091992,1.082186,0.482839,-0.712641,1.174220,0.363720,...,-1.172015,0.240031,-0.064971,-0.734850,-0.066206,-1.034299,0.282986,0.810639,1.162422,0.415489
4,0.0,0.0,0.0,0.804042,-0.560019,0.634758,1.284828,0.081312,-0.033718,1.010153,...,-1.276135,-1.269369,-0.538136,-0.695819,-1.188053,-0.064272,-0.797495,1.021822,1.450210,0.482550


### Execução no terminal

```bash
# 1. Definimos o "espaço de busca" (as opções que queremos testar)
param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],       # Número de árvores
    'max_depth': [None, 5, 10, 20, 30],             # Profundidade máxima
    'min_samples_split': [2, 5, 10],                # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4],                  # Mínimo de amostras em cada folha
    'max_features': ['sqrt', 'log2', None]         # Quantidade de colunas testadas por árvore
}

# 2. Instanciamos o modelo base
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. Configuramos a busca aleatória
# n_iter=20 significa que ele vai testar 20 combinações aleatórias das opções acima
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,               # Validação cruzada em 5 partes (folds)
    random_state=42,
    n_jobs=-1,          # Usa todo o processador
    scoring='accuracy'  # Métrica de escolha (pode ser 'f1', 'roc_auc', etc.)
)

# 4. Rodamos a busca nos dados de treino
random_search.fit(x_train, y_train)

# --- RESULTADOS ---
print("🏆 Melhores Parâmetros Encontrados:")
print(random_search.best_params_)
```

In [10]:
# 1. Definimos o "espaço de busca" (as opções que queremos testar)
param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],       # Número de árvores
    'max_depth': [None, 5, 10, 20, 30],             # Profundidade máxima
    'min_samples_split': [2, 5, 10],                # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4],                  # Mínimo de amostras em cada folha
    'max_features': ['sqrt', 'log2', None]          # Quantidade de colunas testadas por árvore
}

# 2. Instanciamos o modelo base
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# 3. Configuramos a busca aleatória
# n_iter=20 significa que ele vai testar 20 combinações aleatórias das opções acima
random_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_dist, 
    n_iter=20, 
    cv=5,               # Validação cruzada em 5 partes (folds)
    random_state=42, 
    n_jobs=-1,          # Usa todo o processador
    scoring='accuracy'  # Métrica de escolha (pode ser 'f1', 'roc_auc', etc.)
)

# 4. Rodamos a busca nos dados de treino
random_search.fit(x_train, y_train)

# --- RESULTADOS ---
print("🏆 Melhores Parâmetros Encontrados:")
print(random_search.best_params_)

🏆 Melhores Parâmetros Encontrados:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
